In [8]:
import configparser
import os

# Initialize the config parser
config = configparser.ConfigParser()
config.read('config.ini')

# Current active profile
user_profile = 'Xuting' 

# Correcting the variable names to match your config.ini keys
# These refer to the 'input =' and 'output =' lines in your file
input_base = config[user_profile]['input']
output_base = config[user_profile]['output']

print(f"Reading data from: {input_base}")
print(f"Saving results to: {output_base}")

Reading data from: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData
Saving results to: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData


In [9]:
import pyarrow.parquet as pq
import polars as pl

# --- Paths to your files ---
# We use os.path.join to combine the base directory with the filename
quotes_file = "data_quotes_2023_06_23.parquet"
trades_file = "data_trades_2023_06_23.parquet"

quotes_path = os.path.join(input_base, quotes_file)
trades_path = os.path.join(input_base, trades_file)

# --- Efficiently count rows without loading data ---
# Now quotes_path and trades_path are correctly defined
quotes_rows = pq.ParquetFile(quotes_path).metadata.num_rows
trades_rows = pq.ParquetFile(trades_path).metadata.num_rows

print(f"Quotes rows: {quotes_rows:,}")
print(f"Trades rows: {trades_rows:,}")


Quotes rows: 1,473,898,592
Trades rows: 75,036,920


In [10]:
import pyarrow.parquet as pq
from datetime import datetime

# ---- quotes file ----
# Construct the path using the base directory from your config file
quotes_file = "data_quotes_2023_06_23.parquet"
quotes_path = os.path.join(input_base, quotes_file)

pf_quotes = pq.ParquetFile(quotes_path)
print("🧾 Quotes schema:")
print(pf_quotes.schema_arrow)     # column names and types only

# ---- trades file ----
# Construct the path using the base directory from your config file
trades_file = "data_trades_2023_06_23.parquet"
trades_path = os.path.join(input_base, trades_file)

pf_trades = pq.ParquetFile(trades_path)
print("\n💹 Trades schema:")
print(pf_trades.schema_arrow)

🧾 Quotes schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
BID: double
BIDSIZ: int64
ASK: double
ASKSIZ: int64
QU_COND: string
QU_SEQNUM: int64
NATBBO_IND: string
QU_CANCEL: string
QU_SOURCE: string
SYM_ROOT: string
SYM_SUFFIX: string

💹 Trades schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
SYM_ROOT: string
SYM_SUFFIX: string
TR_SCOND: string
SIZE: int64
PRICE: double
TR_STOP_IND: string
TR_CORR: string
TR_SEQNUM: int64
TR_ID: int64
TR_SOURCE: string
TR_RF: string


In [11]:
#Trades

target_file = os.path.join(input_base, "data_trades_2023_06_23.parquet")
target_output = os.path.join(output_base, "2023_06_23/processed_output_trades/")

#%run polar_try_partitioned.py --FILE_PATH=$target_file --OUTPUT_DIR=$target_output

In [12]:
#Quotes
quotes_input_file = os.path.join(input_base, "data_quotes_2023_06_23.parquet")
quotes_output_dir = os.path.join(output_base, "2023_06_23/processed_output_quotes/")

# Ensure the output directory exists to avoid errors
os.makedirs(quotes_output_dir, exist_ok=True)

#%run polar_try_partitioned.py --FILE_PATH=$quotes_input_file --OUTPUT_DIR=$quotes_output_dir

In [13]:
pl.Config.set_tbl_cols(50) 
pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_rows(200) #polars.Config.tbl_rows = 50

polars.config.Config

In [14]:
# Input file remains the same
trades_input = os.path.join(input_base, "data_trades_2023_06_23.parquet")

# Output file with the "_upper" suffix
trades_output_upper = os.path.join(input_base, "data_trades_2023_06_23_upper.parquet")

#%run makesTradesColsUpper.py --INPUT=$trades_input --OUTPUT=$trades_output_upper

pf_trades = pq.ParquetFile(trades_input)
print("\n💹 Trades schema:")
print(pf_trades.schema_arrow)


💹 Trades schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
SYM_ROOT: string
SYM_SUFFIX: string
TR_SCOND: string
SIZE: int64
PRICE: double
TR_STOP_IND: string
TR_CORR: string
TR_SEQNUM: int64
TR_ID: int64
TR_SOURCE: string
TR_RF: string


In [15]:
from pathlib import Path
import pyarrow.parquet as pq
import os

input_file_upper = os.path.join(input_base, "data_trades_2023_06_23_upper.parquet")
output_dir_clean = os.path.join(output_base, "2023_06_23/processed_output_trades_upper_clean_from_single/")

os.makedirs(output_dir_clean, exist_ok=True)


#%run rewrite2.py --IN_FILE=$input_file_upper --OUT_DIR=$output_dir_clean --BATCH_ROWS=25000


DIR = Path(output_dir_clean)

files = sorted(DIR.glob("*.parquet"))
if not files:
    raise RuntimeError(f"No parquet files found in {DIR}")

# Check the schema of the first generated file to verify results
for f in files[:1]:
    arrow_schema = pq.read_schema(f)
    print(f"File Name: {f.name}")
    print(f"Schema:\n{arrow_schema}")


File Name: chunk_000001.parquet
Schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
SYM_ROOT: string
SYM_SUFFIX: string
TR_SCOND: string
SIZE: int64
PRICE: double
TR_STOP_IND: string
TR_CORR: string
TR_SEQNUM: int64
TR_ID: int64
TR_SOURCE: string
TR_RF: string


In [16]:
trades_clean_dir = os.path.join(output_base, "2023_06_23/processed_output_trades_upper_clean_from_single")
quotes_processed_dir = os.path.join(output_base, "2023_06_23/processed_output_quotes")

analysis_output_file = os.path.join(output_base, "2023_06_23/taq_analysis_output.txt")

#%run query.py --TRADES_UPPER=$trades_clean_dir --QUOTES_OLD=$quotes_processed_dir --OUTPUT_FILE=$analysis_output_file


In [17]:
import os
import polars as pl


quotes_path = os.path.join(input_base, "data_quotes_2023_06_23.parquet")

q = pl.scan_parquet(quotes_path).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(q)

21731288

In [18]:
QUOTES_DIR = os.path.join(output_base, "2023_06_23/processed_output_quotes/")
q = pl.scan_parquet(f"{QUOTES_DIR}/*.parquet").filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
len(q)

21731288

In [19]:
QUOTES_DIR = os.path.join(output_base, "2023_06_23/processed_output_quotes_top50/")
q = pl.scan_parquet(f"{QUOTES_DIR}/*.parquet").filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
len(q)

21731288

In [20]:
trades_path = os.path.join(input_base, "data_trades_2023_06_23.parquet")

# Process data
t = pl.scan_parquet(trades_path).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(t)

553292

In [21]:
trades_upper_path = os.path.join(input_base, "data_trades_2023_06_23_upper.parquet")

t = pl.scan_parquet(trades_upper_path).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(t)

553292

In [22]:
TRADES_DIR = os.path.join(output_base, "2023_06_23/processed_output_trades/")

q = pl.scan_parquet(f"{TRADES_DIR}/*.parquet").filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(q)

553292

In [23]:
import polars as pl

schema = {
    "DATE": pl.Date,
    "TIME_M": pl.Time,
    "EX": pl.Utf8,
    "SYM_ROOT": pl.Utf8,
    "SYM_SUFFIX": pl.Utf8,
    "TR_SCOND": pl.Utf8,
    "SIZE": pl.Int64,
    "PRICE": pl.Float64,
    "TR_STOP_IND": pl.Utf8,
    "TR_CORR": pl.Utf8,
    "TR_SEQNUM": pl.Int64,
    "TR_ID": pl.Int64,
    "TR_SOURCE": pl.Utf8,
    "TR_RF": pl.Utf8,
}

TRADES = os.path.join(input_base, "data_trades_2023_06_23_upper.parquet")

t = pl.scan_parquet(TRADES, schema=schema).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
print(len(t))

t = pl.scan_parquet(TRADES, schema=schema).collect()
print(len(t))

553292
75036920


In [24]:
import polars as pl

schema = {
    "DATE": pl.Date,
    "TIME_M": pl.Time,
    "EX": pl.Utf8,
    "SYM_ROOT": pl.Utf8,
    "SYM_SUFFIX": pl.Utf8,
    "TR_SCOND": pl.Utf8,
    "SIZE": pl.Int64,
    "PRICE": pl.Float64,
    "TR_STOP_IND": pl.Utf8,
    "TR_CORR": pl.Utf8,
    "TR_SEQNUM": pl.Int64,
    "TR_ID": pl.Int64,
    "TR_SOURCE": pl.Utf8,
    "TR_RF": pl.Utf8,
}



# TRADES_CLEAN = os.path.join(output_base, "2023_06_23/processed_output_trades_upper_clean/")
# t_clean_spy = pl.scan_parquet(TRADES_CLEAN, schema=schema).filter(
#   (pl.col("SYM_ROOT") == "SPY") & (pl.col("SYM_SUFFIX").fill_null("") == "")
# ).collect()
# print(f"Clean SPY: {len(t_clean_spy)}")

TRADES_SINGLE = os.path.join(output_base, "2023_06_23/processed_output_trades_upper_clean_from_single/")


t_single_spy = pl.scan_parquet(TRADES_SINGLE).filter(
    (pl.col("SYM_ROOT") == "SPY") & (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
print(f"Single SPY: {len(t_single_spy)}")


t_single_all = pl.scan_parquet(TRADES_SINGLE).collect()
print(f"Single Total: {len(t_single_all)}")

Single SPY: 553292
Single Total: 75036920


In [25]:
%run offending_datatypes.py --TRADES_DIR=/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_trades_upper_clean_from_single

Found 3002 parquet files in /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_trades_upper_clean_from_single
  scanned schemas: 200/3002
  scanned schemas: 400/3002
  scanned schemas: 600/3002
  scanned schemas: 800/3002
  scanned schemas: 1000/3002
  scanned schemas: 1200/3002
  scanned schemas: 1400/3002
  scanned schemas: 1600/3002
  scanned schemas: 1800/3002
  scanned schemas: 2000/3002
  scanned schemas: 2200/3002
  scanned schemas: 2400/3002
  scanned schemas: 2600/3002
  scanned schemas: 2800/3002
  scanned schemas: 3000/3002
  scanned schemas: 3002/3002

=== Columns with >1 dtype across files ===

=== SIZE dtype breakdown ===
SIZE dtype Int64: 3002 files
  examples: ['chunk_000001.parquet', 'chunk_000002.parquet', 'chunk_000003.parquet', 'chunk_000004.parquet', 'chunk_000005.parquet', 'chunk_000006.parquet', 'chunk_000007.parquet', 'chunk_000008.parquet', 'chunk_000009.parquet', 'chunk_000010.parquet']

=== Offending files (SIZE != Int64) ===


In [ ]:
#%run query.py  --TRADES_UPPER="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_trades_upper_clean_from_single/*.parquet" --QUOTES_OLD="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_quotes/*.parquet" --OUTPUT_FILE="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/taq_analysis_output.txt"
trades_clean = os.path.join(output_base, "2023_06_23/processed_output_trades_upper_clean_from_single/*.parquet")
quotes_old = os.path.join(output_base, "2023_06_23/processed_output_quotes/*.parquet")
output_txt = os.path.join(output_base, "2023_06_23/taq_analysis_output.txt")

#%run query.py --TRADES_UPPER=$trades_clean --QUOTES_OLD=$quotes_old --OUTPUT_FILE=$output_txt

[INFO] Deleted existing output file before processing: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/taq_analysis_output.txt

Analysis successfully executed and results written to /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/taq_analysis_output.txt
[INFO] Deleted existing output file before processing: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/taq_analysis_output.txt

Analysis successfully executed and results written to /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/taq_analysis_output.txt


/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:61: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(trades_lf.fetch(n_rows=10), file=f)
/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:62: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(quotes_lf.fetch(n_rows=10), file=f)
/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:61: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(trades_lf.fetch(n_rows=10), file=f)
/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:62: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(quotes_lf.fetch(n_rows=10), file=f)


In [30]:
%run stats.py --TRADES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_trades_upper_clean_from_single/" --QUOTES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_quotes/" --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/stats_out_simple/"
t_dir = os.path.join(output_base, "2023_06_23/processed_output_trades_upper_clean_from_single/")
q_dir = os.path.join(output_base, "2023_06_23/processed_output_quotes/")
o_dir = os.path.join(output_base, "2023_06_23/stats_out_simple/")

%run stats.py --TRADES_DIR=$t_dir --QUOTES_DIR=$q_dir --OUT_DIR=$o_dir

[INFO] Deleted existing OUT_DIR before processing: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/stats_out_simple
Output dir: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/stats_out_simple
[TRADES] scanning 3002 files in /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_trades_upper_clean_from_single …
  → 1/3002 chunk_000001.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 1
  → 2/3002 chunk_000002.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 2
  → 3/3002 chunk_000003.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 2
  → 4/3002 chunk_000004.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 20
  → 5/3002 chunk_000005.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 20
  → 6/3002 chunk_000006.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 25
  → 7/3002 chunk_000007.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 28
  → 8/3002 chunk_000008.parquet: 1

/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/stats.py:95: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df = pl.DataFrame(rows, schema=["SYM_ROOT", "SYM_SUFFIX", "TRADES_ROWS", "TRADES_VOL"])



[QUOTES] scanning 11895 files in /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_quotes …
  → 1/11895 chunk_0001.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 124,784
  → 2/11895 chunk_0002.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 247,765
  → 3/11895 chunk_0003.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 372,416
  → 4/11895 chunk_0004.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 497,069
  → 5/11895 chunk_0005.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 620,272
  → 6/11895 chunk_0006.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 744,173
  → 7/11895 chunk_0007.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 867,888
  → 8/11895 chunk_0008.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 992,129
  → 9/11895 chunk_0009.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 1,116,407
  → 10/11895 chunk_0010.parquet: 2 row-groups
    RG 2/2 … cumu

/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/stats.py:145: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df = pl.DataFrame(rows, schema=["SYM_ROOT", "SYM_SUFFIX", "QUOTES_ROWS"])


    RG 1/1 … unique symbols so far: 20
  → 6/3002 chunk_000006.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 25
  → 7/3002 chunk_000007.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 28
  → 8/3002 chunk_000008.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 28
  → 9/3002 chunk_000009.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 31
  → 10/3002 chunk_000010.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 31
  → 11/3002 chunk_000011.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 31
  → 12/3002 chunk_000012.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 31
  → 13/3002 chunk_000013.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 31
  → 14/3002 chunk_000014.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 31
  → 15/3002 chunk_000015.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 31
  → 16/3002 chunk_000016.parquet: 1 row-groups
    RG 1/1 … unique symbols so far: 31
  → 17/3002 chunk_0

/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/stats.py:95: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df = pl.DataFrame(rows, schema=["SYM_ROOT", "SYM_SUFFIX", "TRADES_ROWS", "TRADES_VOL"])



[QUOTES] scanning 11895 files in /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_quotes …
  → 1/11895 chunk_0001.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 124,784
  → 2/11895 chunk_0002.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 247,765
  → 3/11895 chunk_0003.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 372,416
  → 4/11895 chunk_0004.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 497,069
  → 5/11895 chunk_0005.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 620,272
  → 6/11895 chunk_0006.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 744,173
  → 7/11895 chunk_0007.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 867,888
  → 8/11895 chunk_0008.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 992,129
  → 9/11895 chunk_0009.parquet: 2 row-groups
    RG 2/2 … cumulative QUOTES rows: 1,116,407
  → 10/11895 chunk_0010.parquet: 2 row-groups
    RG 2/2 … cumu

/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/stats.py:145: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df = pl.DataFrame(rows, schema=["SYM_ROOT", "SYM_SUFFIX", "QUOTES_ROWS"])


In [31]:
OUT_DIR = os.path.join(output_base, "2023_06_23/stats_out_simple")
TRADES_OUT_CSV = os.path.join(OUT_DIR, "top50_trades_by_volume.csv")
QUOTES_OUT_CSV = os.path.join(OUT_DIR, "top50_quotes_by_rows.csv")

print(f"Reading {TRADES_OUT_CSV} and {QUOTES_OUT_CSV} …")
trades_df = pl.read_csv(TRADES_OUT_CSV)
quotes_df = pl.read_csv(QUOTES_OUT_CSV)

print("\n🔥 Top Trades (Sorted by TRADES_ROWS):")
print(trades_df.sort("TRADES_ROWS", descending=True))

print("\n📊 Top Quotes (Sorted by QUOTES_ROWS):")
print(quotes_df.sort("QUOTES_ROWS", descending=True))

Reading /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/stats_out_simple/top50_trades_by_volume.csv and /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/stats_out_simple/top50_quotes_by_rows.csv …

🔥 Top Trades (Sorted by TRADES_ROWS):
shape: (50, 4)
┌──────────┬────────────┬─────────────┬────────────┐
│ SYM_ROOT ┆ SYM_SUFFIX ┆ TRADES_ROWS ┆ TRADES_VOL │
│ ---      ┆ ---        ┆ ---         ┆ ---        │
│ str      ┆ str        ┆ i64         ┆ i64        │
╞══════════╪════════════╪═════════════╪════════════╡
│ TSLA     ┆            ┆ 1712283     ┆ 185027405  │
│ SPY      ┆            ┆ 553292      ┆ 97842263   │
│ AMZN     ┆            ┆ 534815      ┆ 80503335   │
│ AMD      ┆            ┆ 502056      ┆ 78513314   │
│ TQQQ     ┆            ┆ 362778      ┆ 113092377  │
│ META     ┆            ┆ 344849      ┆ 66497581   │
│ PIK      ┆            ┆ 295124      ┆ 92253145   │
│ MARA     ┆            ┆ 270877      ┆ 92621974   │
│ PLTR     ┆            ┆ 2

In [32]:
# %run top50_trades_corresponding_quotes.py --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/stats_out_simple/" --TRADES_TOP50_CSV=top50_trades_by_volume.csv --TRADES_SRC_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_trades_upper_clean_from_single/" --TRADES_OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_trades_upper_top50/" --QUOTES_SRC_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_quotes/" --QUOTES_OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_quotes_top50/" --MERGED_OUT_CSV="merged_top50_sorted.csv"
# import os


# stats_dir = os.path.join(output_base, "2023_06_23/stats_out_simple/")
# top50_csv = "top50_trades_by_volume.csv"

# trades_src = os.path.join(output_base, "2023_06_23/processed_output_trades_upper_clean_from_single/")
# trades_out = os.path.join(output_base, "2023_06_23/processed_output_trades_upper_top50/")

# quotes_src = os.path.join(output_base, "2023_06_23/processed_output_quotes/")
# quotes_out = os.path.join(output_base, "2023_06_23/processed_output_quotes_top50/")

# os.makedirs(trades_out, exist_ok=True)
# os.makedirs(quotes_out, exist_ok=True)

# # --- 2. Run the Top 50 Processing Script ---
# %run top50_trades_corresponding_quotes.py \
#     --OUT_DIR=$stats_dir \
#     --TRADES_TOP50_CSV=$top50_csv \
#     --TRADES_SRC_DIR=$trades_src \
#     --TRADES_OUT_DIR=$trades_out \
#     --QUOTES_SRC_DIR=$quotes_src \
#     --QUOTES_OUT_DIR=$quotes_out \
#     --MERGED_OUT_CSV="merged_top50_sorted.csv"

In [33]:
# %run top50_trades_corresponding_quotes_persist.py --TRADES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_trades_upper_top50/" --QUOTES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/processed_output_quotes_top50/" --STATS_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/stats_out_simple/" --TOP50_CSV="top50_trades_by_volume.csv" --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/merged_output_top50/" --OUT_FILE="merged_all.parquet" --CHUNK_SIZE=25000

# t_dir = os.path.join(output_base, "2023_06_23/processed_output_trades_upper_top50/")
# q_dir = os.path.join(output_base, "2023_06_23/processed_output_quotes_top50/")
# s_dir = os.path.join(output_base, "2023_06_23/stats_out_simple/")
# out_dir = os.path.join(output_base, "2023_06_23/merged_output_top50/")

# %run top50_trades_corresponding_quotes_persist.py \
#    --TRADES_DIR=$t_dir \
#    --QUOTES_DIR=$q_dir \
#    --STATS_DIR=$s_dir \
#    --TOP50_CSV="top50_trades_by_volume.csv" \
#    --OUT_DIR=$out_dir \
#    --OUT_FILE="merged_all.parquet" \
#    --CHUNK_SIZE=25000

In [34]:
%run ms.py --MERGED_FILE="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/merged_output_top50/merged_all.parquet" --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2023_06_23/ms_out/" --TICKERS=AMZN,SPY,TSLA


merged_file = os.path.join(output_base, "2023_06_23/merged_output_top50/merged_all.parquet")
ms_out_dir = os.path.join(output_base, "2023_06_23/ms_out/")

%run ms.py --MERGED_FILE=$merged_file --OUT_DIR=$ms_out_dir --TICKERS=AMZN,SPY,TSLA

Null TR_SEQNUM : 0
Null QU_SEQNUM : 184033
Zero BID and ASK : 7409

🔎 Running microstructure analysis for: AMZN
shape: (1, 19)
┌────────┬────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ TICKER ┆ N_ROWS ┆ TOTAL_SIZE ┆ VWAP       ┆ AVG_SPREAD ┆ AVG_DEPTH_ ┆ IS_MEAN_BP ┆ IS_MEDIAN_ ┆ IS_MIN_BPS ┆ IS_MAX_BPS ┆ MI_MEAN_BP ┆ MI_MEDIAN_ ┆ MI_MIN_BPS ┆ MI_MAX_BPS ┆ MR_MEAN_BP ┆ MR_MEDIAN_ ┆ MR_MIN_BPS ┆ MR_MAX_BPS ┆ FWD_TRADES │
│ ---    ┆ ---    ┆ ---        ┆ ---        ┆ _BPS       ┆ TOB        ┆ S          ┆ BPS        ┆ ---        ┆ ---        ┆ S          ┆ BPS        ┆ ---        ┆ ---        ┆ S          ┆ BPS        ┆ ---        ┆ ---        ┆ _FOR_MR    │
│ str    ┆ u32    ┆ i64        ┆ f64        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ f64        ┆ f64        ┆ ---        ┆ ---        ┆ 

In [35]:
summary_df

TICKER,N_ROWS,TOTAL_SIZE,VWAP,AVG_SPREAD_BPS,AVG_DEPTH_TOB,IS_MEAN_BPS,IS_MEDIAN_BPS,IS_MIN_BPS,IS_MAX_BPS,MI_MEAN_BPS,MI_MEDIAN_BPS,MI_MIN_BPS,MI_MAX_BPS,MR_MEAN_BPS,MR_MEDIAN_BPS,MR_MIN_BPS,MR_MAX_BPS,FWD_TRADES_FOR_MR
str,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""AMZN""",526582,69720391,129.754763,4090.320216,6.007235,13.113371,0.386593,0.0,2341.78118,0.0,0.0,-0.0,0.0,-10.896545,-0.385579,-2220.835664,1896.110939,10
"""SPY""",551239,94202355,433.84248,2613.023571,10.282239,2.491773,0.115275,0.0,1832.063146,0.0,0.0,-0.0,0.0,-2.758121,-1.3114e-12,-1829.885683,1548.388581,10
"""TSLA""",1698472,183563141,257.682635,9106.714388,7.131097,12.664856,0.781082,0.0,4768.723548,0.0,0.0,-0.0,0.0,-12.705717,-0.583385,-1856.795641,1855.169035,10
